# Import libraries

In [6]:
import geopandas as gpd
import numpy as np
from shapely.geometry import box

# Import file

In [7]:
aoi = gpd.read_file("/content/drive/MyDrive/Agri_RS_AI_Project/Projects/01_NDVI_Classification_Stability/data/raw/AOI/sylhet_aoi.geojson")

print(aoi.crs)

EPSG:4326


# CRS to Projected(meter based)

In [8]:
aoi = aoi.to_crs(epsg=32646)

# GENERATE GRID

a FULL rectangular grid covering Sylhet bounding box

In [9]:
minx, miny, maxx, maxy = aoi.total_bounds

# CREATE GRID FUNCTION

In [10]:
def create_grid(minx, miny, maxx, maxy, cell_size=200):
    grids = []

    x_coords = np.arange(minx, maxx, cell_size)
    y_coords = np.arange(miny, maxy, cell_size)

    id_counter = 1

    for x in x_coords:
        for y in y_coords:
            grid_cell = box(x, y, x + cell_size, y + cell_size)
            grids.append({
                "Grid_ID": f"G{id_counter:06d}",
                "geometry": grid_cell
            })
            id_counter += 1

    return gpd.GeoDataFrame(grids, crs=aoi.crs)

# GENERATE GRID

In [11]:
grid = create_grid(minx, miny, maxx, maxy, cell_size=200)

print(len(grid))
print(grid.head())

142462
   Grid_ID                                           geometry
0  G000001  POLYGON ((361777.834 2720746.389, 361777.834 2...
1  G000002  POLYGON ((361777.834 2720946.389, 361777.834 2...
2  G000003  POLYGON ((361777.834 2721146.389, 361777.834 2...
3  G000004  POLYGON ((361777.834 2721346.389, 361777.834 2...
4  G000005  POLYGON ((361777.834 2721546.389, 361777.834 2...


# CLIP GRID TO AOI (VERY IMPORTANT)

The extended area are croped.

In [12]:
grid_clipped = gpd.overlay(grid, aoi, how="intersection")

# RESET GRID IDs

In [13]:
grid_clipped = grid_clipped.reset_index(drop=True)

grid_clipped["Grid_ID"] = [
    f"G{i:06d}" for i in range(1, len(grid_clipped) + 1)
]

# FINAL CHECK

In [14]:
print(grid_clipped.shape)
print(grid_clipped.head())

(87986, 15)
   Grid_ID                    id  ADM0_CODE   ADM0_NAME  ADM1_CODE ADM1_NAME  \
0  G000001  0001000000000000155a         23  Bangladesh        580    Sylhet   
1  G000002  0001000000000000155a         23  Bangladesh        580    Sylhet   
2  G000003  0001000000000000155a         23  Bangladesh        580    Sylhet   
3  G000004  0001000000000000155a         23  Bangladesh        580    Sylhet   
4  G000005  0001000000000000155a         23  Bangladesh        580    Sylhet   

   ADM2_CODE ADM2_NAME DISP_AREA  EXP2_YEAR        STATUS  STR2_YEAR  \
0       5824    Sylhet        NO       3000  Member State       1000   
1       5824    Sylhet        NO       3000  Member State       1000   
2       5824    Sylhet        NO       3000  Member State       1000   
3       5824    Sylhet        NO       3000  Member State       1000   
4       5824    Sylhet        NO       3000  Member State       1000   

   Shape_Area  Shape_Leng                                           geomet

# SAVE OUTPUT

In [16]:
grid_clipped.to_file(
    "/content/drive/MyDrive/Agri_RS_AI_Project/Projects/01_NDVI_Classification_Stability/data/raw/grid/grid_100m.shp"
)

In [ ]:
grid_clipped.to_file(
    "/content/drive/MyDrive/Agri_RS_AI_Project/Projects/01_NDVI_Classification_Stability/data/raw/grid/grid_100m.shp",
    driver="GeoJSON"
)